# SatcomLLM: Cloud GPU RAG Pipeline Demo

Welcome to the **Cloud GPU** SatcomLLM RAG Demo! This notebook is a copy of the fully local demo, adapted so that **inference runs on a remote GPU** while **document chunks stay on your machine**.

Both the embedding model and the LLM are assumed to already be hosted on the cloud GPU behind OpenAI-compatible HTTP APIs (for example vLLM for chat and vLLM or Text Embeddings Inference for embeddings).

## What's Different from the Other Demos?

| Component | Fully Local Demo | Cloud Services Demo | **This Demo** |
|-----------|------------------|---------------------|---------------|
| **Embeddings** | sentence-transformers (local) | DeepInfra API | **Cloud GPU embedding endpoint** |
| **Vector DB** | Qdrant Local | Qdrant Cloud | **Qdrant Local** |
| **Chunk storage** | Local JSON files | In-memory / cloud | **Local JSON files** |
| **LLM Inference** | Local transformers | RunPod vLLM | **Cloud GPU chat endpoint** |

## What You'll Learn

This notebook demonstrates how to build a RAG system that:
- Sends embedding and generation requests to a GPU already hosting the models
- Stores processed document chunks locally so you can reuse them
- Stores vectors in a local Qdrant instance
- Generates answers with the remote LLM using retrieved local context

## Notebook Structure

The demo is divided into 4 main parts:

- **Setup and LLM Fundamentals**: Install dependencies, configure the GPU endpoints, and test chat plus embeddings.
- **Theory around RAG**: Understand how retrieval complements generation.
- **Embedding Vector Database Creation**: Load and chunk markdown documents locally, embed them on the GPU, and store vectors in local Qdrant. **Chunks are saved locally for reuse.**
- **Retrieval and RAG Pipeline**: Search locally, then generate answers on the cloud GPU.

## Technology Stack

- **Embeddings**: OpenAI-compatible `/v1/embeddings` on the cloud GPU
- **Vector Database**: Qdrant Local (in-memory or persistent)
- **LLM**: OpenAI-compatible `/v1/chat/completions` on the cloud GPU
- **Document Processing**: LangChain's header-based chunking
- **Chunk Storage**: Local JSON files in `./chunks_cache/`

## Prerequisites

- Python 3.8 or higher
- Network access to the GPU host
- Both models already running on that GPU
- Copy `env.example` to `.env` and set the endpoint URLs
- The complete demo takes approximately 5-10 minutes to run

---

# PART 1: Setup & LLM Fundamentals

Before building the RAG system, set up the environment and confirm the GPU endpoints respond.

## What's in This Section?

### 1. Dependency Installation
Install the packages needed on **this** machine. You do **not** need PyTorch or local model weights:
- `requests`: talk to the GPU HTTP APIs
- `langchain`: RAG orchestration
- `qdrant-client`: local vector database
- `python-dotenv`: environment management

### 2. Cloud GPU Configuration
Point the notebook at the hosted services:
- **Embedding Model**: already served on the GPU
- **LLM Model**: already served on the GPU
- **Vector Database**: Qdrant local instance (in-memory or file-based)

### 3. LLM Concepts
- **Tokenization**: Text → Numbers (handled on the GPU server)
- **Prompting**: Crafting instructions
- **Generation**: Controlling output

### 4. Endpoint Testing
Verify chat and embeddings before building RAG.

---

### 1.1. Setup Instructions

The cell below will automatically:
1. **Create a virtual environment** (`./venv/`) if it doesn't exist
2. **Install all required packages** into the venv
3. **Add the venv to Python path** so packages are available in this notebook

**First run**: This will take 1-3 minutes to download and install packages.
**Subsequent runs**: This will be much faster (packages are already installed).

Run the cell below to set up the environment:

In [3]:
import subprocess
import sys
import os
from pathlib import Path
import shutil

# Define venv path
VENV_PATH = Path("./venv")
VENV_PYTHON = VENV_PATH / "bin" / "python"
VENV_PIP = VENV_PATH / "bin" / "pip"

def find_system_python():
    """Find a working system Python 3 to create the venv."""
    python_candidates = [
        "/usr/bin/python3",
        "/usr/local/bin/python3",
        "/opt/homebrew/bin/python3",
        shutil.which("python3"),
    ]

    for python_path in python_candidates:
        if python_path and Path(python_path).exists():
            try:
                result = subprocess.run(
                    [python_path, "--version"],
                    capture_output=True, text=True, timeout=5
                )
                if result.returncode == 0 and "Python 3" in result.stdout:
                    return python_path
            except Exception:
                continue
    return sys.executable

# Step 1: Check if venv exists and is valid
venv_valid = False
if VENV_PATH.exists():
    if VENV_PYTHON.exists():
        try:
            result = subprocess.run(
                [str(VENV_PYTHON), "--version"],
                capture_output=True, text=True, timeout=5
            )
            venv_valid = result.returncode == 0
        except Exception:
            venv_valid = False

    if not venv_valid:
        print("Existing venv is broken, removing it...")
        shutil.rmtree(VENV_PATH, ignore_errors=True)

# Step 2: Create virtual environment if needed
if not VENV_PATH.exists():
    print("Creating virtual environment...")
    system_python = find_system_python()
    print(f"  Using Python: {system_python}")
    subprocess.run([system_python, "-m", "venv", str(VENV_PATH)], check=True)
    print(f"Virtual environment created at {VENV_PATH}")
else:
    print(f"Virtual environment already exists at {VENV_PATH}")

# Step 3: Upgrade pip in venv
print("\nUpgrading pip...")
subprocess.run([str(VENV_PIP), "install", "--upgrade", "pip", "-q"], check=True)

# Step 4: Install required packages (no local model runtime)
print("\nInstalling required packages (this may take a few minutes on first run)...")
packages = [
    "requests",
    "qdrant-client",
    "langchain",
    "langchain-community",
    "langchain-text-splitters",
    "tqdm",
    "python-dotenv",
]

subprocess.run([str(VENV_PIP), "install", "-q"] + packages, check=True)
# Repair native pydantic wheels if a previous openai install left them broken
subprocess.run(
    [str(VENV_PIP), "install", "--force-reinstall", "-q", "pydantic-core", "pydantic"],
    check=True,
)
print("All packages installed successfully!")

# Step 5: Add venv to Python path for this notebook session
venv_site_packages = None
for python_dir in (VENV_PATH / "lib").glob("python*"):
    sp = python_dir / "site-packages"
    if sp.exists():
        venv_site_packages = sp
        break

if venv_site_packages and str(venv_site_packages) not in sys.path:
    sys.path.insert(0, str(venv_site_packages))
    print(f"Added {venv_site_packages} to Python path")

print("\nEnvironment setup complete! You can now run the following cells.")

Virtual environment already exists at venv

Upgrading pip...

Installing required packages (this may take a few minutes on first run)...
All packages installed successfully!

Environment setup complete! You can now run the following cells.


In [4]:
# Ensure venv is in path (in case this cell is run after kernel restart)
import sys
from pathlib import Path

VENV_PATH = Path("./venv")
for python_dir in (VENV_PATH / "lib").glob("python*"):
    sp = python_dir / "site-packages"
    if sp.exists() and str(sp) not in sys.path:
        sys.path.insert(0, str(sp))

# Verify imports work correctly
try:
    import requests
    import langchain
    import qdrant_client
    from dotenv import load_dotenv
    print("All required packages are installed correctly")
    print(f"requests version: {requests.__version__}")
except ImportError as e:
    print(f"Missing package: {e}")
    print("\nPlease run the setup cell above first")
    print("This will create a virtual environment and install all packages.")

All required packages are installed correctly
requests version: 2.34.2


### 1.2 Configure Cloud GPU Endpoints

The notebook reads **`.env`**, not `env.example`. `env.example` is only a template (`python-dotenv` ignores it unless we load it as a fallback).

In this folder:

```bash
cp env.example .env
```

Then edit `.env`. If the notebook runs **on the same GPU pod** as `vllm serve`, use loopback:

```bash
GPU_LLM_URL=http://127.0.0.1:8000/v1
GPU_EMBEDDING_URL=http://127.0.0.1:8002/v1
LLM_MODEL_NAME=esa-sceva/llama3-satcom-8b
EMBEDDING_MODEL_NAME=Qwen/Qwen3-Embedding-4B
```

If the notebook runs on your laptop, use the RunPod proxy URLs instead (`https://<POD_ID>-8000.proxy.runpod.net/v1` and `...-8002...`). After saving `.env`, re-run the configuration cell below. That cell is what defines `GPU_EMBEDDING_URL` and `EMBEDDING_MODEL_NAME` for later cells.

The notebook talks to **OpenAI-compatible** HTTP APIs. That is the usual interface when you host models with vLLM, Text Generation Inference, or Text Embeddings Inference.

**Required settings:**

1. **`GPU_LLM_URL`**: chat completions base URL, including `/v1`  
   Example: `http://203.0.113.10:8000/v1` or `https://<POD_ID>-8000.proxy.runpod.net/v1`
2. **`GPU_EMBEDDING_URL`**: embeddings base URL, including `/v1`  
   On a RunPod pod this is usually a **second** port (`8002`), not `8000` or `8001`. See the hosting section below.
3. **`GPU_API_KEY`**: optional. Leave empty if the server does not check auth
4. **`LLM_MODEL_NAME`** / **`EMBEDDING_MODEL_NAME`**: names as registered on the GPU server

**Assumptions:**
- The LLM is already loaded on the GPU
- The embedding model is already loaded on the GPU
- You do not download or run those models in this notebook

Chunk files and the vector database stay local (`./chunks_cache/`, `./qdrant_db/`).

### Hosting both models on a RunPod GPU with vLLM

This notebook expects **OpenAI-compatible** URLs, not RunPod serverless (`https://api.runpod.ai/v2/.../run`). Rent a **Pod**, expose HTTP ports `8000` and `8002`, then start two `vllm serve` processes (LLM first, embeddings second).


#### Install vLLM (on the GPU pod, not in this notebook)

Do **not** install vLLM in the laptop `./venv` used by this notebook. Install it only on the RunPod machine that will run the models.

**Preferred: official image (avoids most install errors).** Create the pod with container image `vllm/vllm-openai:latest` (or a pinned tag). That image already has CUDA, PyTorch, and vLLM. Skip pip entirely and go to the `vllm serve` commands below.

**If you install with pip**, use a **fresh** environment and the **prebuilt wheel**. Do not build from source (that compiles CUDA kernels and is where installs usually fail).

```bash
# On the GPU pod. Python 3.10–3.12. Check the driver first:
nvidia-smi

# Fresh env (do not reuse a notebook/transformers venv)
python3 -m venv ~/vllm-env
source ~/vllm-env/bin/activate
pip install -U pip uv

# Prebuilt wheel + matching PyTorch/CUDA. --torch-backend=auto picks the index
# from the installed driver so you do not mix cu118 / cu121 / cu124 wheels.
uv pip install vllm --torch-backend=auto
```

Confirm the import before serving:

```bash
python -c "import vllm; print(vllm.__version__)"
vllm serve --help
```

**Avoid these (common failure modes):**
- `pip install vllm` into an existing env that already has `torch`, `transformers`, or `openai` (broken `pydantic_core` / mismatched CUDA)
- `pip install git+https://github.com/vllm-project/vllm` or `pip install -e .` without `VLLM_USE_PRECOMPILED=1` (long source build, `nvcc` errors)
- Python 3.9 (this laptop venv is 3.9; vLLM wants 3.10+)
- Installing on macOS/CPU — vLLM needs the NVIDIA GPU pod

If `uv pip install vllm --torch-backend=auto` still fails, pin the CUDA index that matches `nvidia-smi` (for example CUDA 12.4 → `uv pip install vllm --torch-backend=cu124`) or switch to the Docker image.

Activate the env in every pod terminal (`vllm` is not on `PATH` otherwise):

```bash
source ~/vllm-env/bin/activate
```

One process serves one model. Use [esa-sceva/llama3-satcom-8b](https://huggingface.co/esa-sceva/llama3-satcom-8b) for generation and [Qwen/Qwen3-Embedding-4B](https://huggingface.co/Qwen/Qwen3-Embedding-4B) for embeddings. Start the LLM first and wait until `curl http://127.0.0.1:8000/v1/models` succeeds, then start embeddings in a second terminal.

```bash
# Terminal 1 — LLM (chat completions)
VLLM_USE_FLASHINFER_SAMPLER=0 vllm serve esa-sceva/llama3-satcom-8b \
  --host 0.0.0.0 \
  --port 8000 \
  --gpu-memory-utilization 0.70 \
  --served-model-name esa-sceva/llama3-satcom-8b

# Terminal 2 — embeddings (only after the LLM is healthy)
vllm serve Qwen/Qwen3-Embedding-4B \
  --host 0.0.0.0 \
  --port 8002 \
  --runner pooling \
  --gpu-memory-utilization 0.28 \
  --max-model-len 8192 \
  --served-model-name Qwen/Qwen3-Embedding-4B
```

`--host 0.0.0.0` is required so the RunPod proxy can reach the servers. Binding to `127.0.0.1` typically returns **502**.

#### Ports (do not use 8001)

RunPod's nginx already listens on **8001** (and often 8081) inside the container. `vllm serve --port 8001` fails with `OSError: [Errno 98] Address already in use`. Serve embeddings on **8002** and add **8002** as an HTTP port in the pod UI so `https://<POD_ID>-8002.proxy.runpod.net` is reachable.

#### vLLM 0.29 embedding flags

`--task embed` is not a `vllm serve` option anymore. For embedding / rerank / reward models use `--runner pooling`. vLLM may log `Resolved --convert auto to --convert embed`; you can pass `--convert embed` explicitly to silence that.

#### FlashInfer on Blackwell (SM 12.x)

On cards such as **RTX PRO 6000 Blackwell** the chat server can load weights and then die during sampler warmup with:

```text
RuntimeError: FlashInfer requires GPUs with sm75 or higher
```

That message is misleading: the GPU is SM 12.0, which is far above sm75. FlashInfer's JIT looks at the **system** CUDA toolkit (often `/usr/local/cuda` → 12.8), not PyTorch's CUDA 13.x. SM 12.x needs CUDA **≥ 12.9**. The real warning appears earlier as `Failed to get device capability: SM 12.x requires CUDA >= 12.9`, then the empty arch list becomes `sm75 or higher`.

Workaround for the **LLM** process: `VLLM_USE_FLASHINFER_SAMPLER=0` (uses PyTorch sampling; quality is unchanged). The embedding server does not sample, so this flag is unnecessary there.

#### Split GPU memory (two processes, one GPU)

`--gpu-memory-utilization` is a fraction of **total** VRAM, not leftover VRAM. The default `0.92` lets the first process reserve almost the whole GPU. The second then fails with:

```text
ValueError: Free memory on device cuda:0 (7.52/94.97 GiB) on startup is less than desired GPU memory utilization (0.92, 87.37 GiB)
```

You cannot fix that by only lowering the embedding flag: a 4B bf16 model needs more than ~8 GiB just for weights, and the LLM will already have taken the rest. Restart the LLM with a smaller reservation, then start embeddings.

On a ~95 GiB GPU the commands above use about **66 GiB** for the 8B chat model and **27 GiB** for the 4B embedder. Adjust the two fractions so they add to less than ~0.90 and each model still fits (8B chat ≈ 15 GiB weights; 4B embed ≈ 8 GiB weights).

**Also cap embedding context.** Qwen3-Embedding-4B defaults to `max_model_len` **40960**. With `--gpu-memory-utilization 0.18` the weights load (~7.6 GiB) and then KV-cache init fails:

```text
Available KV cache memory: -33.95 GiB
ValueError: No available memory for the cache blocks. Try increasing `gpu_memory_utilization`
```

`--gpu-memory-utilization` is a budget for weights **plus** activations **plus** KV cache. A 40k context blows that budget. For this RAG demo, `--max-model-len 8192` is enough for document chunks, and `0.28` leaves room for KV blocks. Do not raise the embedder above leftover free VRAM (`nvidia-smi`); if the startup check fails, lower the LLM fraction first.

Copy `env.example` to `.env` and set the proxy URLs (Pod ID is on the pod page / Connect tab):

```bash
GPU_LLM_URL=https://<POD_ID>-8000.proxy.runpod.net/v1
GPU_EMBEDDING_URL=https://<POD_ID>-8002.proxy.runpod.net/v1
GPU_API_KEY=
LLM_MODEL_NAME=esa-sceva/llama3-satcom-8b
EMBEDDING_MODEL_NAME=Qwen/Qwen3-Embedding-4B
```

`LLM_MODEL_NAME` and `EMBEDDING_MODEL_NAME` must match `--served-model-name`. If you pass `--api-key mysecret` to `vllm serve`, set the same value in `GPU_API_KEY`.

Check that both servers are ready:

```bash
# On the pod
curl http://127.0.0.1:8000/v1/models
curl http://127.0.0.1:8002/v1/models

# From the laptop (after exposing 8000 and 8002 as HTTP ports)
curl https://<POD_ID>-8000.proxy.runpod.net/v1/models
curl https://<POD_ID>-8002.proxy.runpod.net/v1/models
```

An 8B chat model plus a 4B embedder needs enough VRAM for both (or two pods). Do not point both URLs at the same `vllm serve` unless that process actually implements `/v1/embeddings`.


### Understanding What Stays Local vs What Runs on the GPU

#### Runs on the cloud GPU
- Embedding the query and every document chunk
- Tokenization of prompts (inside the serving stack)
- LLM generation

#### Stays on this machine
- Source markdown files in `data/`
- Processed chunks in `./chunks_cache/`
- Vector index in local Qdrant (`:memory:` or `./qdrant_db/`)
- Retrieval / similarity search over those local vectors

```
Your Project/
├── .env                           # GPU URLs and model names
├── data/                          # Your source markdown files
│   └── *.md
├── chunks_cache/                  # Saved processed chunks (local)
│   └── <document_name>_chunks.json
└── qdrant_db/                     # Vector database (if using persistent storage)
    └── collections/
        └── satcom_rag_gpu/
```

The GPU never stores your chunk files. It only receives the texts you send for embedding or generation.

In [7]:
# Ensure venv is in path
import sys
from pathlib import Path

VENV_PATH = Path("./venv")
for python_dir in (VENV_PATH / "lib").glob("python*"):
    sp = python_dir / "site-packages"
    if sp.exists() and str(sp) not in sys.path:
        sys.path.insert(0, str(sp))

import os
import requests
from dotenv import load_dotenv

# env.example = defaults / template. .env = your real URLs (overrides).
# You must still have a .env; placeholder your-gpu-host values are rejected below.
_nb_dir = Path(".").resolve()
load_dotenv(_nb_dir / "env.example")
_env_path = _nb_dir / ".env"
if _env_path.exists():
    load_dotenv(_env_path, override=True)
    print(f"Loaded environment from {_env_path}")
else:
    print("No .env file found. Copy env.example to .env and set the GPU URLs.")

# Storage paths stay local
CHUNKS_CACHE_DIR = Path("./chunks_cache")
CHUNKS_CACHE_DIR.mkdir(exist_ok=True)

# Cloud GPU endpoints (OpenAI-compatible HTTP APIs)
GPU_LLM_URL = os.getenv("GPU_LLM_URL", "").rstrip("/")
GPU_EMBEDDING_URL = os.getenv("GPU_EMBEDDING_URL", "").rstrip("/")
GPU_API_KEY = os.getenv("GPU_API_KEY", "").strip()

LLM_MODEL_NAME = os.getenv("LLM_MODEL_NAME", "esa-sceva/llama3-satcom-8b")
EMBEDDING_MODEL_NAME = os.getenv("EMBEDDING_MODEL_NAME", "Qwen/Qwen3-Embedding-4B")


def _require_url(name, value):
    if not value or "your-gpu-host" in value:
        raise ValueError(
            f"{name} is not set to a real endpoint. In this folder run:\n"
            "  cp env.example .env\n"
            "Then edit .env. On this GPU pod use http://127.0.0.1:8000/v1 and "
            "http://127.0.0.1:8002/v1. From a laptop use the RunPod proxy URLs."
        )
    return value


GPU_LLM_URL = _require_url("GPU_LLM_URL", GPU_LLM_URL)
GPU_EMBEDDING_URL = _require_url("GPU_EMBEDDING_URL", GPU_EMBEDDING_URL)


def gpu_headers():
    headers = {"Content-Type": "application/json"}
    if GPU_API_KEY:
        headers["Authorization"] = f"Bearer {GPU_API_KEY}"
    return headers


print("Configuration:")
print(f"  LLM endpoint:        {GPU_LLM_URL}")
print(f"  Embedding endpoint:  {GPU_EMBEDDING_URL}")
print(f"  LLM model:           {LLM_MODEL_NAME}")
print(f"  Embedding model:     {EMBEDDING_MODEL_NAME}")
print(f"  API key set:         {'yes' if GPU_API_KEY else 'no'}")
print(f"  Chunks cache:        {CHUNKS_CACHE_DIR.resolve()}")
print("\nModels are expected to already be loaded on the GPU. This notebook only calls the APIs.")

ValueError: GPU_LLM_URL is not set. Copy env.example to .env and set the GPU endpoint URLs.

## 1.3 Tokenization: Breaking Text into Tokens

Before a language model can process text, the text must be converted into numerical tokens. On this setup that step happens **on the GPU server** inside the hosted serving stack. This notebook never loads a local tokenizer or model weights.

What you send over HTTP is raw text. The server tokenizes it, runs the model, and returns generated text (plus optional token-usage metadata).

In [4]:
sample_text = "Satellite communications enable global connectivity through geostationary and low Earth orbit satellites."

print("Original text:")
print(f"  {sample_text}")
print(f"\nCharacter count: {len(sample_text)}")
print(f"Whitespace-separated words: {len(sample_text.split())}")
print(
    "\nThe GPU serving stack will tokenize this string with the hosted model's "
    "vocabulary. Token counts from the API (if returned) are the source of truth."
)

Original text:
  Satellite communications enable global connectivity through geostationary and low Earth orbit satellites.

Character count: 105
Whitespace-separated words: 12

The GPU serving stack will tokenize this string with the hosted model's vocabulary. Token counts from the API (if returned) are the source of truth.


## 1.4 Cloud GPU LLM Inference

Now let's call the hosted LLM. The helper below uses the OpenAI-compatible chat completions API that vLLM and similar servers expose.

In [ ]:
def chat_with_gpu_llm(prompt, max_tokens=256, temperature=0.2, top_p=0.9):
    """
    Generate text using the LLM hosted on the cloud GPU.

    Args:
        prompt: Input text prompt
        max_tokens: Maximum tokens to generate
        temperature: Sampling temperature
        top_p: Top-p sampling parameter

    Returns:
        Generated text
    """
    response = requests.post(
        f"{GPU_LLM_URL}/chat/completions",
        headers=gpu_headers(),
        json={
            "model": LLM_MODEL_NAME,
            "messages": [{"role": "user", "content": prompt}],
            "max_tokens": max_tokens,
            "temperature": temperature,
            "top_p": top_p,
        },
        timeout=120,
    )
    response.raise_for_status()
    payload = response.json()
    return (payload["choices"][0]["message"]["content"] or "").strip()


print("Cloud GPU LLM client ready")
print(f"  POST {GPU_LLM_URL}/chat/completions")
print(f"  model={LLM_MODEL_NAME}")

In [ ]:
print("Testing cloud GPU LLM inference...\n")
test_prompt = "What is satellite communications in one sentence?"
print(f"Question: {test_prompt}\n")

response = chat_with_gpu_llm(test_prompt, max_tokens=128, temperature=0.7)
print(f"Response: {response}\n")
print("Cloud GPU LLM inference working correctly!")

---

# PART 2: THEORY AROUND RAG SYSTEM

Now that the hosted model answers questions, let's understand how to implement RAG with **custom documents**.

## 2.1. The Problem with Standard LLMs

Large Language Models have limitations:

| Issue | Description | Impact |
|-------|-------------|--------|
| **Hallucinations** | Generate plausible but false info | Unreliable answers |
| **Knowledge Cutoff** | Training data has a date limit | Outdated information |
| **No Source** | Can't cite where info came from | Unverifiable |
| **Generic** | Lack domain-specific expertise | Poor specialized answers |
| **Static** | Can't update without retraining | Expensive to maintain |

## How RAG Solves These Problems

**Retrieval-Augmented Generation** adds a knowledge retrieval step:

```
Standard LLM:
Question → LLM → Answer (may hallucinate)

RAG System:
Question → Find Relevant Docs → LLM + Context → Grounded Answer
```

### Key Benefits:

- Grounded: Answers based on actual documents
- Verifiable: Shows sources used
- Up-to-date: Update docs without retraining model
- Domain-specific: Add specialized knowledge
- Cost-effective: Cheaper than fine-tuning

## 2.2 RAG Architecture

### Two Main Phases:

#### Phase 1: Indexing (One-Time Setup)
```
Documents → Clean → Chunk (local) → Embed (GPU) → Store in local Vector DB
```

#### Phase 2: Query (Every Question)
```
Question → Embed (GPU) → Search local Vector DB → Retrieve Docs →
    Augment Prompt → LLM (GPU) → Answer
```

## 2.3 Architecture Overview

```
┌─────────────┐
│   User      │ Asks a question
│  Question   │
└──────┬──────┘
       ↓
┌──────────────────────────────────────────┐
│  1. EMBED QUERY (Cloud GPU embeddings)   │
│     Convert question → vector            │
└──────┬───────────────────────────────────┘
       ↓
┌──────────────────────────────────────────┐
│  2. SEMANTIC SEARCH (Local Qdrant)       │
│     Find top-k similar document chunks   │
└──────┬───────────────────────────────────┘
       ↓
┌──────────────────────────────────────────┐
│  3. RETRIEVE CONTEXT                     │
│     Get text + metadata from matches     │
└──────┬───────────────────────────────────┘
       ↓
┌──────────────────────────────────────────┐
│  4. ENRICH PROMPT                        │
│     Question + Retrieved Context         │
└──────┬───────────────────────────────────┘
       ↓
┌──────────────────────────────────────────┐
│  5. GENERATE ANSWER (Cloud GPU LLM)      │
│     Hosted model produces grounded reply │
└──────┬───────────────────────────────────┘
       ↓
┌──────────────┐
│   Answer     │ With source attribution
│ + Sources    │
└──────────────┘
```

Let's start building!

---

# Part 3: Embedding & Vector Database

In this part, we'll build the knowledge base:

1. **Load documents** from local files (Markdown).
2. **Chunk the documents** intelligently to preserve semantic structure.
3. **Save chunks locally** to avoid re-processing.
4. **Generate embeddings** for each chunk on the cloud GPU.
5. **Upload the embeddings to a local Qdrant database**, making them ready for retrieval.

## 3.1. Document Loading and Chunking

Let's load and chunk our documents. **Chunks will be saved locally for reuse!**

In [12]:
import os
from pathlib import Path
import json

# Load all markdown documents from the data folder
data_folder = Path("data")

# Check if data folder exists
if not data_folder.exists():
    print(f"Data folder not found: {data_folder}")
    print("Creating sample data folder...")
    data_folder.mkdir(exist_ok=True)
    print("Please add your markdown files to the 'data' folder")
    documents = {}
else:
    # Find all markdown files
    markdown_files = list(data_folder.glob("*.md")) + list(data_folder.glob("*.markdown"))

    if not markdown_files:
        print(f"No markdown files found in {data_folder}")
        print("Please add markdown files to the 'data' folder")
        documents = {}
    else:
        print(f"Found {len(markdown_files)} markdown file(s) in '{data_folder}':")
        for f in markdown_files:
            print(f"  - {f.name}")

        # Load all documents
        documents = {}
        total_chars = 0

        for file_path in markdown_files:
            with open(file_path, "r", encoding="utf-8") as f:
                content = f.read()
                documents[file_path.name] = content
                total_chars += len(content)
                print(f"\n{file_path.name}: {len(content)} characters")

        print(f"\nTotal characters across all documents: {total_chars}")
        if documents:
            print("\nPreview of first document:")
            first_doc = list(documents.values())[0]
            print(first_doc[:500])

Found 1 markdown file(s) in 'data':
  - dace33f5-f959-4955-bd68-00229c97599e.md

dace33f5-f959-4955-bd68-00229c97599e.md: 26787 characters

Total characters across all documents: 26787

Preview of first document:
# Analyzing Multispectral Satellite Imagery of South American Wildfires Using Deep Learning

[PERSON]

_Monta Vista High School_

Cupertino, CA, United States

###### Abstract

Since frequent severe droughts are lengthening the dry season in the Amazon Rainforest, it is important to detect wildfires promptly and forecast possible spread for effective suppression response. Current wildfire detection models are not versatile enough for the low-technology conditions of South American hot spots. Thi


In [14]:
import json
from pathlib import Path
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter
from langchain_core.documents import Document

# Local cache path (this cell does not need the GPU config cell)
if "CHUNKS_CACHE_DIR" not in globals():
    CHUNKS_CACHE_DIR = Path("./chunks_cache")
CHUNKS_CACHE_DIR.mkdir(exist_ok=True)

def chunk_markdown_document(markdown_text, max_chunk_size=1000, chunk_overlap=200):
    """
    Chunk markdown document using header-based splitting.

    Args:
        markdown_text: Markdown formatted text
        max_chunk_size: Maximum chunk size in characters
        chunk_overlap: Overlap between chunks for context preservation

    Returns:
        List of Document objects with content and metadata
    """
    headers_to_split_on = [
        ("#", "Header 1"),
        ("##", "Header 2"),
        ("###", "Header 3"),
        ("####", "Header 4"),
    ]

    markdown_splitter = MarkdownHeaderTextSplitter(
        headers_to_split_on=headers_to_split_on
    )
    md_header_splits = markdown_splitter.split_text(markdown_text)

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=max_chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
    )

    all_chunks = []
    for idx, doc in enumerate(md_header_splits):
        if len(doc.page_content) > max_chunk_size:
            sub_chunks = text_splitter.split_text(doc.page_content)
            for sub_idx, sub_chunk in enumerate(sub_chunks):
                chunk_doc = Document(
                    page_content=sub_chunk,
                    metadata={
                        **doc.metadata,
                        "chunk_id": f"{idx}_{sub_idx}",
                        "chunk_size": len(sub_chunk),
                    },
                )
                all_chunks.append(chunk_doc)
        else:
            doc.metadata["chunk_id"] = str(idx)
            doc.metadata["chunk_size"] = len(doc.page_content)
            all_chunks.append(doc)

    return all_chunks

def save_chunks_to_file(chunks, filename):
    """Save chunks to a JSON file for later reuse."""
    chunks_data = []
    for chunk in chunks:
        chunks_data.append({
            "page_content": chunk.page_content,
            "metadata": chunk.metadata,
        })

    filepath = CHUNKS_CACHE_DIR / filename
    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(chunks_data, f, indent=2, ensure_ascii=False)

    return filepath

def load_chunks_from_file(filename):
    """Load chunks from a saved JSON file."""
    filepath = CHUNKS_CACHE_DIR / filename
    if not filepath.exists():
        return None

    with open(filepath, "r", encoding="utf-8") as f:
        chunks_data = json.load(f)

    chunks = []
    for chunk_data in chunks_data:
        chunk = Document(
            page_content=chunk_data["page_content"],
            metadata=chunk_data["metadata"],
        )
        chunks.append(chunk)

    return chunks

# Chunk all documents (with local caching)
if "documents" in locals() and documents:
    all_chunks = []

    for filename, markdown_text in documents.items():
        cache_filename = f"{Path(filename).stem}_chunks.json"
        cached_chunks = load_chunks_from_file(cache_filename)

        if cached_chunks:
            print(f"\n✓ Loading cached chunks for {filename} ({len(cached_chunks)} chunks)")
            all_chunks.extend(cached_chunks)
        else:
            print(f"\nProcessing {filename}...")
            doc_chunks = chunk_markdown_document(markdown_text)

            for chunk in doc_chunks:
                chunk.metadata["source_file"] = filename

            saved_path = save_chunks_to_file(doc_chunks, cache_filename)
            print(f"  Generated {len(doc_chunks)} chunks")
            print(f"  ✓ Saved to {saved_path}")

            all_chunks.extend(doc_chunks)

    print(f"\n{'=' * 60}")
    print(f"Total chunks across all documents: {len(all_chunks)}")
    print(f"{'=' * 60}")

    if all_chunks:
        print("\nSample chunks:")
        for i, chunk in enumerate(all_chunks[:3]):
            print(f"\n--- Chunk {i + 1} ---")
            print(f"Source: {chunk.metadata.get('source_file', 'unknown')}")
            print(f"Headers: {chunk.metadata.get('Header 1', '')} > {chunk.metadata.get('Header 2', '')}")
            print(f"Content preview: {chunk.page_content[:150]}...")

    chunks = all_chunks
else:
    print("⚠ No documents loaded. Please add markdown files to the 'data' folder.")
    chunks = []


Processing dace33f5-f959-4955-bd68-00229c97599e.md...
  Generated 43 chunks
  ✓ Saved to chunks_cache/dace33f5-f959-4955-bd68-00229c97599e_chunks.json

Total chunks across all documents: 43

Sample chunks:

--- Chunk 1 ---
Source: dace33f5-f959-4955-bd68-00229c97599e.md
Headers: Analyzing Multispectral Satellite Imagery of South American Wildfires Using Deep Learning > 
Content preview: [PERSON]  
_Monta Vista High School_  
Cupertino, CA, United States  
###### Abstract...

--- Chunk 2 ---
Source: dace33f5-f959-4955-bd68-00229c97599e.md
Headers: Analyzing Multispectral Satellite Imagery of South American Wildfires Using Deep Learning > 
Content preview: Since frequent severe droughts are lengthening the dry season in the Amazon Rainforest, it is important to detect wildfires promptly and forecast poss...

--- Chunk 3 ---
Source: dace33f5-f959-4955-bd68-00229c97599e.md
Headers: Analyzing Multispectral Satellite Imagery of South American Wildfires Using Deep Learning > 
Content preview

### Chunk Caching Explained

The chunking process includes **local caching**. This is unchanged from the fully local demo.

#### How It Works:

1. **Check Cache**: Before processing, checks if chunks exist in `./chunks_cache/<filename>_chunks.json`
2. **Load from Cache**: If found, loads instantly (no re-processing)
3. **Process & Save**: If not found, processes the document and saves chunks to cache
4. **Reuse**: Next time you run the notebook, chunks load from cache

Cache files store text and metadata only. Embeddings are generated later by the GPU and stored in local Qdrant.

#### Managing Cache:

- **Clear cache**: Delete files from `./chunks_cache/`
- **Force re-process**: Delete a specific cache file or the entire folder

## 3.2 Initialize Cloud GPU Embeddings

Now we'll call the **embedding model already hosted on the GPU**. The wrapper below uses the OpenAI-compatible embeddings API.

In [ ]:
class CloudGPUEmbeddings:
    """Embeddings via an OpenAI-compatible endpoint on the cloud GPU."""

    def __init__(self, base_url, model):
        self.base_url = base_url.rstrip("/")
        self.model = model

    def embed_documents(self, texts):
        if not texts:
            return []
        response = requests.post(
            f"{self.base_url}/embeddings",
            headers=gpu_headers(),
            json={"model": self.model, "input": list(texts)},
            timeout=120,
        )
        response.raise_for_status()
        data = sorted(response.json()["data"], key=lambda item: item.get("index", 0))
        return [item["embedding"] for item in data]

    def embed_query(self, text):
        return self.embed_documents([text])[0]


print(f"Connecting to embedding model on GPU: {EMBEDDING_MODEL_NAME}")
embedding_model = CloudGPUEmbeddings(GPU_EMBEDDING_URL, EMBEDDING_MODEL_NAME)

sample_text = "The SatcomLLM pipeline generates synthetic QA pairs from documents"
sample_embedding = embedding_model.embed_query(sample_text)

print("✓ Cloud GPU embeddings are reachable")
print(f"✓ Model: {embedding_model.model}") 
print(f"✓ Embedding dimension: {len(sample_embedding)}")
print(f"✓ Sample embedding (first 10 values): {sample_embedding[:10]}")

NameError: name 'EMBEDDING_MODEL_NAME' is not defined

## 3.3 Create Local Qdrant Vector Database

Time to set up the **local vector database** where we'll store embeddings returned by the GPU.

### Collection Configuration

| Parameter | Value | Explanation |
|-----------|-------|-------------|
| **Name** | `satcom_rag_gpu` | Identifier for this knowledge base |
| **Vector Size** | detected from GPU embedding | Must match the hosted embedding model |
| **Distance Metric** | Cosine | Best for normalized embeddings |
| **Storage** | Local (in-memory or file-based) | Chunks and vectors stay on this machine |

### Vector Database Storage Options

Qdrant can store data in two ways. Persistent file-based storage is enabled below.

#### Option 1: In-Memory (Fast but Temporary)
```python
qdrant_client = QdrantClient(":memory:")
```

#### Option 2: Persistent File-Based (Recommended)
```python
qdrant_client = QdrantClient(path="./qdrant_db")
```

If you previously ran the fully local notebook against the same `./qdrant_db/` folder, this demo uses a **different collection name** so the two indexes do not collide.

In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct
import uuid

print("Initializing local Qdrant vector database...")
# qdrant_client = QdrantClient(":memory:")  # In-memory storage (fast, but not persistent)
qdrant_client = QdrantClient(path="./qdrant_db")

collection_name = "satcom_rag_gpu"
embedding_dim = len(embedding_model.embed_query("test"))

print(f"Creating collection: {collection_name}")
print(f"Vector dimension (from GPU embedding model): {embedding_dim}")

try:
    qdrant_client.create_collection(
        collection_name=collection_name,
        vectors_config=VectorParams(size=embedding_dim, distance=Distance.COSINE),
    )
    print(f"Collection '{collection_name}' created successfully!")
except Exception as e:
    if "already exists" in str(e).lower():
        print(f"Collection '{collection_name}' already exists, using existing collection")
    else:
        raise

### Understanding the Embedding & Upload Process

1. **Load chunks** from `./chunks_cache/` (or chunk fresh documents)
2. **Send chunk text to the GPU** in batches for embedding
3. **Store the returned vectors locally** in Qdrant together with the original text

```
Local cached chunks → GPU embeddings → Local Qdrant points
```

In [ ]:
from tqdm import tqdm

if chunks:
    collection_info = qdrant_client.get_collection(collection_name=collection_name)
    existing_points = collection_info.points_count

    if existing_points > 0:
        print(f"ℹ Collection '{collection_name}' already contains {existing_points} points.")
        print("  Skipping upload - using existing data.")
        print("  (To re-upload: delete ./qdrant_db/ folder and restart)")
        SKIP_UPLOAD = True
    else:
        SKIP_UPLOAD = False

    if not SKIP_UPLOAD:
        print(f"Embedding {len(chunks)} chunks on the cloud GPU and uploading to local Qdrant...")
        print("This may take a moment...\n")

        points = []
        batch_size = 32
        for i in tqdm(range(0, len(chunks), batch_size), desc="Embedding chunks on GPU"):
            batch_chunks = chunks[i:i + batch_size]
            texts = [chunk.page_content for chunk in batch_chunks]
            embeddings = embedding_model.embed_documents(texts)

            for j, (chunk, embedding) in enumerate(zip(batch_chunks, embeddings)):
                point_id = str(uuid.uuid4())
                points.append(
                    PointStruct(
                        id=point_id,
                        vector=list(embedding),
                        payload={
                            "text": chunk.page_content,
                            "metadata": chunk.metadata,
                            "chunk_id": chunk.metadata.get("chunk_id", f"{i + j}"),
                            "source": chunk.metadata.get("source_file"),
                        },
                    )
                )

        print(f"\nUploading {len(points)} points to local Qdrant...")
        qdrant_client.upsert(
            collection_name=collection_name,
            points=points,
        )
        print(f"✓ Successfully uploaded {len(points)} chunks to local Qdrant!")

    collection_info = qdrant_client.get_collection(collection_name=collection_name)
    print("\nCollection info:")
    print(f"  - Points count: {collection_info.points_count}")
    print(f"  - Vector size: {collection_info.config.params.vectors.size}")
else:
    print("⚠ No chunks to upload. Please load documents first.")

# Part 4: Retrieval, Prompt Enrichment, and Generation

The knowledge base is local. Query embeddings and final answers come from the GPU.

## 4.1: Implement Semantic Search

### How Semantic Search Works

1. **Query Embedding (GPU)**: your question is sent to the hosted embedding model
2. **Vector Search (local Qdrant)**: cosine similarity over stored chunk vectors
3. **Result Retrieval (local)**: matching text and metadata are read from Qdrant payload

In [ ]:
def search_knowledge_base(query, top_k=3):
    """
    Search the local knowledge base. Only the query embedding is computed on the GPU.

    Args:
        query: User's question
        top_k: Number of top results to return

    Returns:
        List of relevant documents with scores
    """
    query_embedding = embedding_model.embed_query(query)

    search_results = qdrant_client.query_points(
        collection_name=collection_name,
        query=query_embedding,
        limit=top_k,
    )

    results = []
    for result in search_results.points:
        results.append({
            "text": result.payload["text"],
            "metadata": result.payload["metadata"],
            "score": result.score,
        })

    return results

if chunks:
    test_query = "What is SatcomLLM?"
    print(f"Test Query: {test_query}\n")
    print("=" * 80)

    results = search_knowledge_base(test_query, top_k=3)

    for i, result in enumerate(results, 1):
        print(f"\n--- Result {i} (Score: {result['score']:.4f}) ---")
        print(f"Section: {result['metadata'].get('Header 1', '')} > {result['metadata'].get('Header 2', '')}")
        print(f"Text preview: {result['text'][:300]}...")

    print("\n" + "=" * 80)
    print("✓ Search function working correctly!")
else:
    print("⚠ No chunks available for search. Please load documents first.")

## 4.2 Complete RAG Pipeline

Retrieval stays local. Generation uses the LLM hosted on the cloud GPU.

In [ ]:
def ask_rag_question(question, top_k=3, max_tokens=256, temperature=0.7, show_sources=True):
    """
    RAG pipeline: retrieve local chunks, generate the answer on the cloud GPU.

    Args:
        question: User's question
        top_k: Number of documents to retrieve
        max_tokens: Maximum tokens for generation
        temperature: Sampling temperature
        show_sources: Whether to display retrieved sources

    Returns:
        Generated answer
    """
    print(f"\n{'=' * 80}")
    print(f"Question: {question}")
    print(f"{'=' * 80}\n")

    print("Searching local knowledge base...")
    retrieved_docs = search_knowledge_base(question, top_k=top_k)

    if not retrieved_docs:
        return "No relevant documents found in the knowledge base."

    if show_sources:
        print(f"\nRetrieved {len(retrieved_docs)} relevant documents:")
        for i, doc in enumerate(retrieved_docs, 1):
            headers = doc["metadata"].get("Header 1", "")
            if doc["metadata"].get("Header 2"):
                headers += f" > {doc['metadata'].get('Header 2')}"
            print(f"  [{i}] {headers} (score: {doc['score']:.3f})")

    context_parts = []
    for i, doc in enumerate(retrieved_docs, 1):
        headers = []
        if doc["metadata"].get("Header 1"):
            headers.append(doc["metadata"]["Header 1"])
        if doc["metadata"].get("Header 2"):
            headers.append(doc["metadata"]["Header 2"])

        section_info = " > ".join(headers) if headers else "General"
        context_parts.append(f"[Document {i}] {section_info}\n{doc['text']}\n")

    context = "\n".join(context_parts)

    enriched_prompt = f"""You are a helpful assistant specializing in satellite communications.

Use the following context from the documentation to answer the question accurately and concisely.
If the answer cannot be found in the context, say so clearly.

CONTEXT:
{context}

QUESTION: {question}

ANSWER:"""

    print("\nGenerating answer with the cloud GPU LLM...")
    answer = chat_with_gpu_llm(
        enriched_prompt,
        max_tokens=max_tokens,
        temperature=temperature,
    )

    print(f"\n{'─' * 80}")
    print("ANSWER:")
    print(f"{'─' * 80}")
    print(answer)
    print(f"\n{'=' * 80}\n")

    return answer

if chunks:
    print("Testing Complete Cloud GPU RAG Pipeline")
    print("=" * 80)

    ask_rag_question(
        "What is SatcomLLM?",
        top_k=3,
        temperature=0.7,
        max_tokens=256,
    )
else:
    print("⚠ No chunks available. Please load documents first.")

## 4.3 Testing with Custom Questions

Let's test the hybrid RAG system with several questions:

In [ ]:
if chunks:
    test_questions = [
        "What is a LEO satellite?",
        "What is deep learning?",
        "What is Internet of Things?",
    ]

    for question in test_questions:
        ask_rag_question(question, top_k=3, temperature=0.7, max_tokens=256)
        print("\n" + "=" * 80 + "\n")
else:
    print("⚠ No chunks available. Please load documents first.")

### Storage Management

Only local artifacts use disk on this machine. Model weights live on the GPU host.

In [ ]:
from pathlib import Path

print(f"Chunks cache: {CHUNKS_CACHE_DIR.resolve()}")
if CHUNKS_CACHE_DIR.exists():
    chunk_files = list(CHUNKS_CACHE_DIR.glob("*_chunks.json"))
    chunks_cache_size = sum(f.stat().st_size for f in CHUNKS_CACHE_DIR.rglob("*") if f.is_file())
    print(f"  Files: {len(chunk_files)}")
    print(f"  Size: {chunks_cache_size / (1024 ** 2):.2f} MB")
else:
    print("  (not created yet)")

qdrant_path = Path("./qdrant_db")
print(f"\nLocal Qdrant: {qdrant_path.resolve()}")
if qdrant_path.exists():
    qdrant_size = sum(f.stat().st_size for f in qdrant_path.rglob("*") if f.is_file())
    print(f"  Size: {qdrant_size / (1024 ** 2):.2f} MB")
else:
    print("  (not created yet)")

#### Managing Storage

**To Free Up Space**:
1. **Clear chunks cache**: delete `./chunks_cache/` (chunks will be re-processed)
2. **Clear Qdrant DB**: delete `./qdrant_db/` (vectors will be re-embedded on the GPU)

**To Persist Data Between Sessions**:
1. Use persistent Qdrant storage: `QdrantClient(path="./qdrant_db")`
2. Chunks are automatically cached in `./chunks_cache/`
3. Documents in `data/` are your source files

Model weights are **not** cached here. They stay on the GPU host.

## Summary: Cloud GPU + Local Storage RAG Pipeline

Congratulations! You've built a hybrid RAG system with:

### Components:
1. **Document Processing**: Markdown chunking with header-based splitting (local)
2. **Chunk Caching**: Local JSON files for fast re-loading
3. **Embeddings**: Hosted embedding model on the cloud GPU
4. **Vector Database**: Local Qdrant (in-memory or file-based)
5. **LLM Generation**: Hosted LLM on the cloud GPU
6. **RAG Orchestration**: Retrieve locally, generate remotely

### Why this split:
- **Heavier compute** (embedding + generation) uses the GPU
- **Your documents and chunks** never have to live in a cloud vector database
- **No local GPU or model download** is required on the laptop running the notebook

### Storage Locations Summary:

| Component | Location | Persistent? |
|-----------|----------|-------------|
| **LLM + embedding models** | Cloud GPU host | Yes (on that machine) |
| **Chunks Cache** | `./chunks_cache/*.json` | Yes |
| **Vector DB** | RAM or `./qdrant_db/` | If file-based |
| **Source Docs** | `./data/*.md` | Yes |

### Next Steps:
- Try different questions
- Adjust `top_k` for more or less context
- Experiment with temperature settings
- Add more documents to the knowledge base
- Point `.env` at a larger hosted model if the GPU has one

---